# 1. What you'll learn

        - alternate discriminator and generator updates
- inspect samples and loss dynamics honestly
- measure diversity loss when latent inputs collapse

        **The one question this notebook answers:** How can a generator learn from a discriminator, and what does mode collapse look like numerically?

# 2. Setup

This lesson keeps generative adversarial networks small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
from sklearn.datasets import load_digits
from sklearn.neighbors import KNeighborsClassifier

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Real 8×8 digits scaled to [-1,1] keep adversarial training CPU-friendly. Class labels are hidden from the GAN and used only by a k-NN probe to summarize generated modes.

**Small example:** If a discriminator assigns five fake probabilities [.1,.2,.4,.7,.8], the generator wants all five moved toward 1. A shortcut that creates only one convincing digit can reduce loss while sacrificing diversity.

In [ ]:
d=load_digits();X=(d.data/8-1).astype("float32");y=d.target;print("Real images:",X.shape,"range:",(X.min(),X.max()),"class counts:",np.bincount(y).tolist())

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].imshow(np.hstack([X[i].reshape(8,8) for i in range(8)]),cmap="gray");axes[0].axis("off");axes[0].set_title("Real training examples")
axes[1].hist(X.ravel(),bins=25,color="#176b66");axes[1].set_title("Scaled pixels");axes[2].bar(range(10),np.bincount(y),color="#d38b45");axes[2].set_title("Hidden-from-GAN classes");plt.tight_layout();plt.show()

**Takeaway:** The GAN sees images but no digit labels.

**Takeaway:** Tanh output naturally matches the [-1,1] pixel range.

**Takeaway:** Balanced real modes let generated-mode imbalance reveal collapse rather than source imbalance.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
loader=DataLoader(TensorDataset(torch.tensor(X)),batch_size=96,shuffle=True,generator=torch.Generator().manual_seed(SEED),drop_last=True);probe=KNeighborsClassifier(5).fit(X,y);print("Batches:",len(loader),"batch shape:",next(iter(loader))[0].shape)

# 6. Train

        Training turns the assumptions behind generative adversarial networks into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Draw real images and latent noise.
2. Train the discriminator to score real high and generated low.
3. Freeze its goal conceptually, then train the generator to make fakes score high.
4. Alternate carefully; neither player has a fixed loss landscape.

In [ ]:
torch.manual_seed(SEED);G=nn.Sequential(nn.Linear(16,48),nn.LeakyReLU(.2),nn.Linear(48,64),nn.Tanh());D=nn.Sequential(nn.Linear(64,48),nn.LeakyReLU(.2),nn.Linear(48,1));go=torch.optim.Adam(G.parameters(),lr=.001,betas=(.5,.999));do=torch.optim.Adam(D.parameters(),lr=.001,betas=(.5,.999));gh=[];dh=[]
for epoch in range(32):
 ge=de=0
 for (real,) in loader:
  z=torch.randn(len(real),16);fake=G(z);do.zero_grad();dl=nn.functional.binary_cross_entropy_with_logits(D(real),torch.ones(len(real),1)*.9)+nn.functional.binary_cross_entropy_with_logits(D(fake.detach()),torch.zeros(len(real),1));dl.backward();do.step()
  go.zero_grad();gl=nn.functional.binary_cross_entropy_with_logits(D(G(torch.randn(len(real),16))),torch.ones(len(real),1));gl.backward();go.step();ge+=gl.item();de+=dl.item()
 gh.append(ge/len(loader));dh.append(de/len(loader))
print("Final generator/discriminator loss:",round(gh[-1],3),round(dh[-1],3))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
with torch.no_grad(): generated=G(torch.randn(400,16)).numpy();fixed=G(torch.zeros(64,16)).numpy()
mode=probe.predict(generated);counts=np.bincount(mode,minlength=10);p=counts/counts.sum();entropy=-np.sum(p*np.log(p+1e-12));diversity=np.mean(np.var(generated,axis=0));mean_base=np.repeat(X.mean(0)[None,:],400,axis=0);nearest=np.mean(np.min(((generated[:,None,:]-X[::10][None,:,:])**2).mean(2),axis=1));base_nearest=np.mean(np.min(((mean_base[:,None,:]-X[::10][None,:,:])**2).mean(2),axis=1))
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(gh,label="generator");axes[0].plot(dh,label="discriminator");axes[0].legend();axes[0].set_title("Adversarial losses")
axes[1].imshow(np.block([[generated[r*8+c].reshape(8,8) for c in range(8)] for r in range(8)]),cmap="gray");axes[1].axis("off");axes[1].set_title("64 generated samples")
axes[2].bar(range(10),counts,color="#176b66");axes[2].set_title(f"k-NN mode probe · entropy {entropy:.2f}");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"Pixel diversity={diversity:.3f}; nearest-real error={nearest:.3f}; mean-image baseline={base_nearest:.3f}")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: The generator can map many latent inputs to one convincing mode; adversarial loss may look acceptable while diversity collapses. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
with torch.no_grad(): one_z=torch.randn(1,16).repeat(400,1);collapsed=G(one_z).numpy()
collapsed_mode=probe.predict(collapsed);cp=np.bincount(collapsed_mode,minlength=10)/len(collapsed_mode);centropy=-np.sum(cp*np.log(cp+1e-12))
print(f"Healthy mode entropy={entropy:.3f} of max {np.log(10):.3f}; identical-latent collapse entropy={centropy:.3f}; collapsed pixel variance={np.var(collapsed,axis=0).mean():.6f}")

**Use this when…** you can monitor sample quality/diversity and need fast one-pass generation.

        **Don't use this when…** training stability, likelihood evaluation, full mode coverage, or reproducibility are strict requirements.

        ## Try this

        1. Train discriminator twice per generator step and inspect balance.
2. Raise latent dimension from 4 to 64.
3. Track nearest-neighbour copies to detect memorization.